In [3]:
from dotenv import load_dotenv
import os

hf_token=os.getenv("HF_TOKEN")

In [4]:
!pip install -q requests


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

## Tool Creation

In [6]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [7]:
res=multiply.invoke({'a': 2, 'b': 3})
res

6

In [8]:
multiply.name

'multiply'

In [9]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [10]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

## Tool Binding

In [37]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hf_token
)

model=ChatHuggingFace(llm=llm)

In [38]:
model.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e91fa-aa8a-7fe0-a38d-50236013ab0a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18})

In [39]:
model_with_tools=model.bind_tools([multiply])

In [40]:
model_with_tools.invoke('hi how are you')

AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to assist you. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 321, 'total_tokens': 354}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e91fb-212e-7950-b266-d2493dcc053c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 321, 'output_tokens': 33, 'total_tokens': 354})

## Tool Calling

In [41]:
query=HumanMessage('can you multiply 3 with 30')

In [42]:
msgs=[query]

In [43]:
msgs

[HumanMessage(content='can you multiply 3 with 30', additional_kwargs={}, response_metadata={})]

In [44]:
res=model_with_tools.invoke(msgs)

In [45]:
msgs.append(res)

In [46]:
msgs

[HumanMessage(content='can you multiply 3 with 30', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 30}', 'name': 'multiply', 'description': None}, 'id': 'call_6U5JL0E6mFTWMzzd6pXPy5mh', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 326, 'total_tokens': 352}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e91fc-160c-7560-94b4-ffd55b2cce38-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 30}, 'id': 'call_6U5JL0E6mFTWMzzd6pXPy5mh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 326, 'output_tokens': 26, 'total_tokens': 352})]

In [50]:
res.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 30},
 'id': 'call_6U5JL0E6mFTWMzzd6pXPy5mh',
 'type': 'tool_call'}

In [52]:
tool_res=multiply.invoke(res.tool_calls[0])

In [53]:
tool_res

ToolMessage(content='90', name='multiply', tool_call_id='call_6U5JL0E6mFTWMzzd6pXPy5mh')

In [54]:
msgs.append(tool_res)

In [55]:
msgs

[HumanMessage(content='can you multiply 3 with 30', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 30}', 'name': 'multiply', 'description': None}, 'id': 'call_6U5JL0E6mFTWMzzd6pXPy5mh', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 326, 'total_tokens': 352}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e91fc-160c-7560-94b4-ffd55b2cce38-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 30}, 'id': 'call_6U5JL0E6mFTWMzzd6pXPy5mh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 326, 'output_tokens': 26, 'total_tokens': 352}),
 ToolMessage(content='90', name='multiply', tool_call_id='call_6U5JL0E6mFTWMzzd6pXPy5mh')]

In [57]:
final_res=model_with_tools.invoke(msgs)
final_res

AIMessage(content='The product of 3 and 30 is 90.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 413, 'total_tokens': 427}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9200-7911-7391-8737-30b61a296f4e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 413, 'output_tokens': 14, 'total_tokens': 427})

In [58]:
final_res.content

'The product of 3 and 30 is 90.'